# Task 3: Soft Mixture-of-Experts Restoration (Simple Baseline)

Implements the soft-routed Mixture-of-Experts framework (Plan 5 §Task 3 & Plan 6 §3).
Reuses Task 2a's classifier as the gating network and Task 2b's specialists as experts.

### Stages:
1. **Warm-up**: Freeze experts, train gate only (short schedule, e.g. 5 epochs).
2. **Joint Fine-Tuning**: Unfreeze experts, train with joint loss (reconstruction + classification + load balance regularizer).

In [ ]:
# 1. Config
WARMUP_EPOCHS = 5
FINETUNE_EPOCHS = 15
WARMUP_LR = 1e-3
FINETUNE_LR = 1e-4
BATCH_SIZE = 32
TEMPERATURE = 1.0

# Joint loss lambdas (Plan 6 §3.3)
LAMBDAS = {
    "lambda1": 0.8,   # L1 reconstruction
    "lambda2": 0.2,   # (1 - SSIM)
    "lambda3": 0.1,   # Classification cross-entropy
    "lambda4": 0.01,  # Gating balance regularizer
}
SEED = 42

In [ ]:
import numpy as np
# 2. Imports & Setup
import sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

RESEARCH_ROOT = Path("..").resolve()
if str(RESEARCH_ROOT) not in sys.path:
    sys.path.insert(0, str(RESEARCH_ROOT))

from src.device_utils import get_device, device_report
from src.datasets import PetDataset
from src.models_dae import ConvDAE
from src.models_classifier import CorruptionClassifier, SoftMoERestorer
from src.train_loop import train_one_epoch_moe, validate_moe, set_seed
from src.checkpoint_utils import create_checkpoint_dict, save_checkpoint, load_checkpoint

set_seed(SEED)
device = get_device(verbose=True)
checkpoint_dir = RESEARCH_ROOT / "checkpoints"

In [ ]:
# 3. Load Trained Sub-networks (Task 2a & 2b Checkpoints)
gate = CorruptionClassifier().to(device)
sp_salt = ConvDAE().to(device)
sp_blur = ConvDAE().to(device)
sp_occl = ConvDAE().to(device)

# Load existing weights if checkpoints exist
for name, m, p in [
    ("gate", gate, checkpoint_dir / "task2_classifier_best.pt"),
    ("salt", sp_salt, checkpoint_dir / "task2b_specialist_salt_best.pt"),
    ("blur", sp_blur, checkpoint_dir / "task2b_specialist_blur_best.pt"),
    ("occl", sp_occl, checkpoint_dir / "task2b_specialist_occlusion_best.pt"),
]:
    if not p.exists():
        raise FileNotFoundError(f"Train prerequisite checkpoint first: {p}")
    load_checkpoint(p, m, device=device)

moe = SoftMoERestorer(
    gate=gate,
    specialist_salt=sp_salt,
    specialist_blur=sp_blur,
    specialist_occlusion=sp_occl,
    temperature=TEMPERATURE
).to(device)

In [ ]:
# 4. Data Loading (PetDataset with ground-truth label for joint loss)
train_dataset = PetDataset(mode="train", corruption_mode="all", return_label=True, seed=SEED)
val_dataset = PetDataset(mode="val", corruption_mode="all", return_label=True, seed=SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# 5. Stage 1: Warm-Up Training (Gate Only)
print("--- Stage 1: Gating Warm-Up (Experts Frozen) ---")
# Freeze experts
for p in moe.specialist_salt.parameters(): p.requires_grad = False
for p in moe.specialist_blur.parameters(): p.requires_grad = False
for p in moe.specialist_occlusion.parameters(): p.requires_grad = False
for p in moe.gate.parameters(): p.requires_grad = True

opt_warmup = torch.optim.Adam(moe.gate.parameters(), lr=WARMUP_LR)

for epoch in range(1, WARMUP_EPOCHS + 1):
    tr = train_one_epoch_moe(moe, train_loader, opt_warmup, device, LAMBDAS)
    vl = validate_moe(moe, val_loader, device, LAMBDAS)
    print(f"[Warmup {epoch:02d}] Val Loss: {vl['val_loss']:.4f}, SSIM: {vl['val_ssim']:.4f}, Gate Usage: {np.round(vl['avg_expert_weights'], 3)}")

In [ ]:
# 6. Stage 2: Joint Fine-Tuning (All Parameters Unfrozen)
print("--- Stage 2: Joint Fine-Tuning ---")
for p in moe.parameters():
    p.requires_grad = True

opt_joint = torch.optim.Adam(moe.parameters(), lr=FINETUNE_LR)
best_val_loss = float("inf")

for epoch in range(1, FINETUNE_EPOCHS + 1):
    tr = train_one_epoch_moe(moe, train_loader, opt_joint, device, LAMBDAS)
    vl = validate_moe(moe, val_loader, device, LAMBDAS)
    
    is_best = vl["val_loss"] < best_val_loss
    if is_best:
        best_val_loss = vl["val_loss"]
        
    ckpt = create_checkpoint_dict(moe, opt_joint, epoch, vl["val_loss"], random_seed=SEED)
    save_checkpoint(
        checkpoint_dir=checkpoint_dir,
        prefix="task3_soft_moe",
        ckpt_dict=ckpt,
        is_best=is_best,
        task_name="task3",
        phase_name="simple",
        device_name=device_report()["device_name"]
    )
    print(f"[Joint Epoch {epoch:02d}] Val Loss: {vl['val_loss']:.4f} (SSIM: {vl['val_ssim']:.4f}) {'*Best*' if is_best else ''}")